<a href="https://colab.research.google.com/github/dasmiq/cs6200-documents/blob/main/PageRank.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PageRank

In this exercise, you will compute PageRank on a collection of 469,235 web sites using the iterative version of the PageRank algorithm described in class for sparse graphs.  As we discussed in class, this avoids performing cubic-time operations on a graph with a large number of nodes.

Furthermore, you should take advantage of the _sparsity_ of the original hyperlinks, even though "teleportation" or random jumps connect every note to every other node.  As long as the maximum degree of the original link graph is much less than the number of nodes, you should be able to keep each iteration's runtime less than quadratic in the number of nodes.

Consider the following directed graph:

![A directed link graph](https://www.khoury.northeastern.edu/home/dasmith/courses/cs6200//pagerank.jpg)

We can represent this graph as a collection of nodes, here, ordered pairs of node index and node name:

In [1]:
small_nodes = [(0, 'A'),
              (1, 'B'),
              (2, 'C'),
              (3, 'D'),
              (4, 'E'),
              (5, 'F')]

and a collection of directed links, i.e., ordered pairs from source to target:

In [2]:
small_edges = [
  (0, 1),
  (0, 2),
  (0, 5),
  (1, 2),
  (1, 3),
  (1, 4),
  (1, 5),
  (2, 3),
  (2, 4),
  (3, 0),
  (3, 2),
  (3, 4),
  (3, 5),
  (4, 0),
  (5, 0),
  (5, 1),
  (5, 4)
]

We use integer identifiers for the nodes for efficiency. In most real-world collections of hyperlinks, unlike this example, not every page will have in-links, nor will every page have out-links.

## First Implementation and Test

\[10 points\] Implement the iterative PageRank algorithm we discussed in class. Test your code on the six-node example using the input representation given above.  Be sure that your code handles pages that have no in-links or out-links properly.  (You may wish to test on a few such examples.) In later parts of this assignment, depending on how you store the data, it may be convenient to use iterators rather than storing the data in memory.

In [3]:
# TODO: Implement PageRank, given nodes and edges, to start with a uniform
# distribution over nodes, run a fixed number of iterations, and
# return a distribution over nodes.

import math

lam = 0.15  # λ = probability of a random jump

def out_degrees(nodes, edges):
  # L_v = number of out-links of each page v (p.59)
  out_deg = [0] * len(nodes)
  for s, t in edges:
    out_deg[s] += 1
  return out_deg

def pr_step(pr, edges, out_deg, lam):
  # One PageRank iteration: PR(u) = λ/N + (1-λ) * sum(PR(v)/L_v)
  N = len(pr)
  # T = total PageRank on sinks (no out-links); spread evenly over all pages 
  T = sum(pr[i] for i in range(N) if out_deg[i] == 0)
  new = [lam / N + (1 - lam) * T / N] * N
  # Each source sends its PageRank, split evenly, to each page it links to 
  for s, t in edges:
    new[t] += (1 - lam) * pr[s] / out_deg[s]
  return new

def page_rank_fixed_iter(nodes, edges, iterations=10):
  N = len(nodes)
  out_deg = out_degrees(nodes, edges)
  pr = [1 / N] * N  # uniform start 
  for _ in range(iterations):
    pr = pr_step(pr, edges, out_deg, lam)
  return [(i, name, pr[i]) for i, name in nodes]

# Output PageRank on the toy graph at various points.
# Make sure your output has node number, name, and PageRank value.
for n in [1, 10, 100]:
  result = page_rank_fixed_iter(small_nodes, small_edges, n)
  print(f"--- {n} iteration(s), sum = {sum(p for _, _, p in result):.4f}")
  for i, name, p in result:
    print(i, name, round(p, 4))

# Extra test: A has no in-links (source), C has no out-links (sink)
test_nodes = [(0, 'A'), (1, 'B'), (2, 'C')]
test_edges = [(0, 1), (1, 2)]
result = page_rank_fixed_iter(test_nodes, test_edges, 10)
print(f"--- source/sink test, sum = {sum(p for _, _, p in result):.4f}")
for i, name, p in result:
  print(i, name, round(p, 4))

# Output PageRank on the toy graph at various points.
# Make sure your output has node number, name, and PageRank value.
page_rank_fixed_iter(small_nodes, small_edges, 1)
page_rank_fixed_iter(small_nodes, small_edges, 10)
page_rank_fixed_iter(small_nodes, small_edges, 100)

--- 1 iteration(s), sum = 1.0000
0 A 0.2493
1 B 0.1194
2 C 0.1431
3 D 0.1312
4 E 0.2139
5 F 0.1431
--- 10 iteration(s), sum = 1.0000
0 A 0.252
1 B 0.1393
2 C 0.1513
3 D 0.119
4 E 0.1871
5 F 0.1513
--- 100 iteration(s), sum = 1.0000
0 A 0.2521
1 B 0.1393
2 C 0.1513
3 D 0.1189
4 E 0.187
5 F 0.1513
--- source/sink test, sum = 1.0000
0 A 0.1843
1 B 0.3412
2 C 0.4745


[(0, 'A', 0.25212710537519556),
 (1, 'B', 0.13930618531853856),
 (2, 'C', 0.15130648986670525),
 (3, 'D', 0.11890782257353917),
 (4, 'E', 0.18704590699931606),
 (5, 'F', 0.15130648986670525)]

**Choice of λ = 0.15.** λ is the probability that the random surfer jumps to a random page instead of following a link, and 0.15 is the typical value from the lectures. It is a trade-off: a smaller λ gives the links more say, but takes more iterations to converge and lets loops of pages trap more PageRank; a larger λ converges faster but pushes everything toward the uniform distribution, so the links matter less. 0.15 sits in between. On the toy graph, the values after 10 and 100 iterations already match to three decimals (e.g. A: 0.252 vs 0.2521), so it converges quickly.

**Handling sinks (pages with no out-links).** Without special handling, sinks leak PageRank and the values stop summing to 1. So the PageRank sitting on all sinks (T) is added up and spread evenly over all N pages, which is the same as the surfer always jumping to a random page from a dead end. The other option, deleting dead ends before running PageRank, would change the graph and leave those pages without a score. This way every page keeps a score, and the total stays at 1.0000 after every run above.

**Extra test.** The six-node graph has no sources or sinks, so it cannot catch a sink bug. The three-node graph A → B → C has both: A is a source and C is a sink. The result still sums to 1.0000, C ranks highest (0.4745) since it collects PageRank from the chain, and A ranks lowest (0.1843) since the only PageRank it gets is from random jumps.

## PageRank on Web Crawl Data

\[20 points\] Download and unpack a list of `.edu` websites and the links among them from the [Common Crawl](https://commoncrawl.org/2017/05/hostgraph-2017-feb-mar-apr-crawls/) open-source web crawl. For the sake of brevity, the data record links among websites, not web pages. The information for nodes and links is the same as the toy example above.

In [ ]:
# If you're running on a machine (e.g., Windows) that doesn't have wget or gzip,
# feel free to comment this out and use a different set of commands to load
# the data.
# !wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/vertices-edu.txt.gz
# !gzip -df vertices-edu.txt.gz
# !wget https://khoury.northeastern.edu/home/dasmith/courses/cs6200/edges-edu.txt.gz
# !gzip -df edges-edu.txt.gz

There should now be files `vertices-edu.txt` and `edges-edu.txt`.

In [4]:
# TODO: Process the raw data into the same format as the simple graph.
# You may create lists or iterators.

nodes = []
with open('vertices-edu.txt') as f:
  for line in f:
    i, name = line.split()
    nodes.append((int(i), name))

edges = []
with open('edges-edu.txt') as f:
  for line in f:
    s, t = line.split()
    edges.append((int(s), int(t)))

print(len(nodes), 'nodes,', len(edges), 'edges')

469235 nodes, 3300462 edges


**Lists vs. iterators; sparse update.** `nodes` and `edges` are loaded into lists (469235 nodes, 3300462 edges) instead of re-reading the files with iterators. Lists take more memory, but every iteration goes over all the edges, so reading the files once is much faster than reading them again on every iteration. Each iteration also only loops over the edges that actually exist, instead of building the full N × N table of every possible link, so the cost grows with the number of pages plus links, not pages squared. Both PageRank versions share one update function (`pr_step`), so they are guaranteed to use the same formula.

Refine your implementation of PageRank to test for numerical convergence. Specificially, at each iteration, calculate the [perplexity](https://en.wikipedia.org/wiki/Perplexity) of the PageRank distribution, where perplexity is defined as 2 raised to the [Shannon entropy](https://en.wikipedia.org/wiki/Entropy_(information_theory)) of the PageRank distribution, i.e., $2^{H(PR)}$. (Recall the definition of entropy from our discussion of data compression.) The maximum perplexity of a PageRank distribution will therefore be the number of nodes in the graph.

At each iteration, check the _change_ in perplexity. If the change is less than some threshold, you can stop.


In [5]:
# TODO: Implement convergence testing in PageRank
# If you choose, you can share some subroutines with your first version.
# Print the change in perplexity at each iteration.

def perplexity(pr):
  # perplexity = 2^H, where H = -sum(p * log2 p) is the Shannon entropy
  H = -sum(p * math.log2(p) for p in pr if p > 0)
  return 2 ** H

def page_rank(nodes, edges, threshold=1):
  N = len(nodes)
  out_deg = out_degrees(nodes, edges)
  pr = [1 / N] * N
  perp = perplexity(pr)  # starts at N, the maximum
  iteration = 0
  while True:
    pr = pr_step(pr, edges, out_deg, lam)
    new_perp = perplexity(pr)
    iteration += 1
    change = abs(new_perp - perp)
    print(f"iteration {iteration}: perplexity = {new_perp:.4f}, change = {change:.4f}")
    perp = new_perp
    if change < threshold:
      break
  return [(i, name, pr[i]) for i, name in nodes]

# Run until perplexity changes by less than 1
PR = page_rank(nodes, edges, 1)

iteration 1: perplexity = 298071.4462, change = 171163.5538
iteration 2: perplexity = 255742.8147, change = 42328.6314
iteration 3: perplexity = 234252.2722, change = 21490.5426
iteration 4: perplexity = 225975.9559, change = 8276.3163
iteration 5: perplexity = 221538.9998, change = 4436.9561
iteration 6: perplexity = 219533.9646, change = 2005.0352
iteration 7: perplexity = 218300.9225, change = 1233.0421
iteration 8: perplexity = 217671.9390, change = 628.9835
iteration 9: perplexity = 217238.0146, change = 433.9244
iteration 10: perplexity = 216999.3836, change = 238.6310
iteration 11: perplexity = 216815.2333, change = 184.1504
iteration 12: perplexity = 216708.5297, change = 106.7036
iteration 13: perplexity = 216619.0678, change = 89.4619
iteration 14: perplexity = 216564.5220, change = 54.5458
iteration 15: perplexity = 216516.2282, change = 48.2937
iteration 16: perplexity = 216485.3513, change = 30.8769
iteration 17: perplexity = 216457.0576, change = 28.2938
iteration 18: per

**Convergence.** From the lectures, the stopping rule is when the old and new PageRank values are close, ‖new − old‖ < τ; here the rule is instead to stop when the change in perplexity drops below the given threshold of 1. Perplexity starts at N (everything uniform) and drops as PageRank concentrates on popular sites. A threshold of 1 is actually strict given how large the values are: the run stopped at iteration 34 with a change of 0.9891 on a perplexity of 216350.2488. A larger threshold would stop sooner but be less accurate.

## Link Analysis

\[20 points\] In this final section, you will compute some properties of this web-site graph.

First, consider the _in-link count_ of a website, simply the number of websites pointing to it (including self-links).

In [6]:
# TODO: List the document ID, domain name, and in-link count of the 70 websites with the highest in-link count

in_deg = [0] * len(nodes)
for s, t in edges:
  in_deg[t] += 1

top_inlinks = sorted(range(len(nodes)), key=lambda i: in_deg[i], reverse=True)[:70]
for i in top_inlinks:
  print(i, nodes[i][1], in_deg[i])

185524 edu.mit.web 4388
278032 edu.stanford 4021
244433 edu.purdue.english.owl 3531
140443 edu.indiana 3339
237176 edu.princeton 3251
64587 edu.columbia 3123
465503 edu.yale 2804
418623 edu.utexas 2622
383763 edu.unc 2592
197698 edu.nap 2494
439637 edu.washington 2291
373442 edu.umich 2281
440674 edu.washington.depts 2276
148945 edu.jhu.muse 2255
60975 edu.colorado 2232
449738 edu.wisc 2230
38320 edu.bu 2205
83572 edu.dartmouth 1965
408380 edu.usc 1952
178879 edu.mit 1946
27307 edu.berkeley 1908
233405 edu.pitt 1857
191069 edu.msu 1810
326371 edu.uchicago.press 1763
136464 edu.illinois 1753
93874 edu.educause 1741
56979 edu.cmu.cs 1730
199032 edu.ncsu 1709
36294 edu.brown 1702
202182 edu.nd 1689
68675 edu.cornell 1685
71095 edu.cornell.law 1646
183214 edu.mit.mitpress 1644
215627 edu.nyu 1625
56538 edu.cmu 1583
239378 edu.psu 1541
350412 edu.ufl 1533
120819 edu.harvard 1529
270369 edu.si 1513
107916 edu.gatech 1500
365396 edu.uky 1497
337138 edu.ucop 1482
358246 edu.uic 1472
382564 edu

Then, use the PageRank values compute by your second implementation. Note that some websites will have both a high in-link count and PageRank.

In [7]:
# TODO: List the document ID, domain name, and PageRank of the 70 websites with the highest PageRank.

top_pr = sorted(PR, key=lambda x: x[2], reverse=True)[:70]
for i, name, p in top_pr:
  print(i, name, p)

185524 edu.mit.web 0.0009280045215322738
465503 edu.yale 0.0007005900334393489
318278 edu.ucdavis.cas 0.0006415669573733464
278032 edu.stanford 0.0005944669473432134
136464 edu.illinois 0.0005796990531329204
319795 edu.ucdavis.ice.code 0.0005465311734617455
237176 edu.princeton 0.0005019558567346617
284517 edu.stanford.web 0.0004747836125752742
383763 edu.unc 0.000471677099661261
64587 edu.columbia 0.00044736809874239356
449738 edu.wisc 0.0004362073078533286
60975 edu.colorado 0.0004323956937774718
346969 edu.ucsf 0.00043004968390032884
140443 edu.indiana 0.00042962437894349
42502 edu.byu.cas 0.00041783006127258575
334739 edu.uconn 0.00041278720126005784
14945 edu.arizona 0.00040927569589571113
244433 edu.purdue.english.owl 0.0004030289428300184
373442 edu.umich 0.0004020785091408695
439637 edu.washington 0.00039842523816994904
408380 edu.usc 0.00039145468869639343
191069 edu.msu 0.0003913952221351178
281817 edu.stanford.med 0.0003913865284455904
418623 edu.utexas 0.0003819559079030015

Finally, compute some summary statistics on this dataset.

In [9]:
# TODO: Compute:

N = len(nodes)
out_deg = out_degrees(nodes, edges)

# - the proportion of websites with no in-links (i.e., source nodes);
no_inlinks = sum(1 for d in in_deg if d == 0) / N
print('Proportion with no in-links (sources):', no_inlinks)

# - the proportion of websites with no out-links (i.e., sink nodes);
no_outlinks = sum(1 for d in out_deg if d == 0) / N
print('Proportion with no out-links (sinks):', no_outlinks)

# - the proportion of websites whose PageRank is higher than the initial uniform distribution.
above_uniform = sum(1 for _, _, p in PR if p > 1 / N) / N
print('Proportion with PageRank > 1/N:', above_uniform)


Proportion with no in-links (sources): 0.26153633041013563
Proportion with no out-links (sinks): 0.6106641661427643
Proportion with PageRank > 1/N: 0.15514614212494807
